# 8. Series de Tiempo por Causa

## 8.1. Causa vs Año

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")


Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
tasas = sin_all.groupby(['year','cause_name'])['age_adjusted_death_rate'].mean().reset_index()
causas_lista = sorted(tasas['cause_name'].unique())

from plotly.subplots import make_subplots
n = len(causas_lista)
cols = 2; rows = (n + cols - 1) // cols

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=causas_lista,
                    shared_xaxes=False)
for i, causa in enumerate(causas_lista):
    r, c = divmod(i, cols)
    sub = tasas[tasas['cause_name']==causa].sort_values('year')
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['age_adjusted_death_rate'],
                             mode='lines+markers', name=causa,
                             line=dict(color='#2171B5', width=1.2),
                             marker=dict(size=4), showlegend=False),
                  row=r+1, col=c+1)
fig.update_layout(height=900, title='Evolución de las tasas de mortalidad ajustadas por edad',
                  template='plotly_white')
fig.show()

## 8.2. Heatmap de muertes absolutas por causas y años

In [3]:
hm = (sin_all.groupby(['year','cause_name'])['deaths']
       .sum().reset_index()
       .pivot(index='cause_name', columns='year', values='deaths'))

fig = go.Figure(data=go.Heatmap(
    z=hm.values, x=hm.columns, y=hm.index,
    colorscale=[[0,'#DCE6F1'],[0.5,'#6C8EBF'],[1,'#1A3A5C']],
    colorbar=dict(title='Número de<br>muertes')
))
fig.update_layout(title='Intensidad de muertes absolutas por causa y año',
                  xaxis_title='Año', yaxis_title='',
                  height=500, template='plotly_white')
fig.show()

**Interpretación:** El heatmap evidencia que enfermedades cardíacas y cáncer son las causas con mayor intensidad (celdas más oscuras), manteniéndose como líderes en todo el período. El Alzheimer y las lesiones no intencionales muestran un aumento progresivo en la intensidad.

## 8.3. Análisis Geográfico: Disparidades por Estado

### 8.4. Mapeo – Número de muertes de 1999 a 2017 por estados

In [4]:
state_codes = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","Florida":"FL","Georgia":"GA",
    "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA",
    "Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD",
    "Massachusetts":"MA","Michigan":"MI","Minnesota":"MN","Mississippi":"MS",
    "Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV","New Hampshire":"NH",
    "New Jersey":"NJ","New Mexico":"NM","New York":"NY","North Carolina":"NC",
    "North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR","Pennsylvania":"PA",
    "Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD","Tennessee":"TN",
    "Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA","Washington":"WA",
    "West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY","District of Columbia":"DC"
}

map_df = (df[(df['cause_name']=='All causes')&(df['state']!='United States')]
           .copy())
map_df['code'] = map_df['state'].map(state_codes)
map_df = map_df.dropna(subset=['code'])

frames, sliders_steps = [], []
for yr in sorted(map_df['year'].unique()):
    sub = map_df[map_df['year']==yr]
    frames.append(go.Frame(
        data=[go.Choropleth(locations=sub['code'], z=sub['deaths'],
                            locationmode='USA-states',
                            colorscale=[[0,'#f7fbff'],[0.5,'#6baed6'],[1,'#08306b']],
                            zmin=map_df['deaths'].min(), zmax=map_df['deaths'].max(),
                            text=sub['state']+'<br>Muertes: '+sub['deaths'].apply(lambda x: f"{x:,}"),
                            hoverinfo='text', colorbar=dict(title='Muertes'))],
        name=str(int(yr))))
    sliders_steps.append(dict(args=[[str(int(yr))],{"frame":{"duration":600},"mode":"immediate"}],
                              label=str(int(yr)), method="animate"))

first = map_df[map_df['year']==1999]
fig = go.Figure(
    data=[go.Choropleth(locations=first['code'], z=first['deaths'],
                        locationmode='USA-states',
                        colorscale=[[0,'#f7fbff'],[0.5,'#6baed6'],[1,'#08306b']],
                        text=first['state'], hoverinfo='text+z',
                        colorbar=dict(title='Muertes'))],
    frames=frames
)
fig.update_layout(
    title='Número de muertes por estado — EE.UU. (1999–2017)',
    geo=dict(scope='usa', showlakes=False),
    height=500,
    sliders=[dict(active=0, steps=sliders_steps, y=0, len=0.9,
                  currentvalue=dict(prefix="Año: "))],
    updatemenus=[dict(type='buttons', showactive=False, y=0.05, x=0.05,
                      buttons=[dict(label="▶ Reproducir",
                                    method="animate",
                                    args=[None,{"frame":{"duration":600},"fromcurrent":True}])])]
)
fig.show()

### 8.5. Mapeo – Tasa de mortalidad ajustada por edad de 1999 a 2017 por estados

In [5]:
rate_df = map_df.copy()
frames2, sliders2 = [], []
for yr in sorted(rate_df['year'].unique()):
    sub = rate_df[rate_df['year']==yr]
    frames2.append(go.Frame(
        data=[go.Choropleth(locations=sub['code'], z=sub['age_adjusted_death_rate'],
                            locationmode='USA-states',
                            colorscale=[[0,'#fff5f0'],[0.5,'#fc8d59'],[1,'#7f0000']],
                            zmin=rate_df['age_adjusted_death_rate'].min(),
                            zmax=rate_df['age_adjusted_death_rate'].max(),
                            text=sub['state'], hoverinfo='text+z',
                            colorbar=dict(title='Tasa<br>ajustada'))],
        name=str(int(yr))))
    sliders2.append(dict(args=[[str(int(yr))],{"frame":{"duration":600},"mode":"immediate"}],
                         label=str(int(yr)), method="animate"))

first2 = rate_df[rate_df['year']==1999]
fig2 = go.Figure(
    data=[go.Choropleth(locations=first2['code'], z=first2['age_adjusted_death_rate'],
                        locationmode='USA-states',
                        colorscale=[[0,'#fff5f0'],[0.5,'#fc8d59'],[1,'#7f0000']],
                        text=first2['state'], hoverinfo='text+z',
                        colorbar=dict(title='Tasa ajustada'))],
    frames=frames2
)
fig2.update_layout(
    title='Tasa de mortalidad ajustada por estado — EE.UU. (1999–2017)',
    geo=dict(scope='usa', showlakes=False), height=500,
    sliders=[dict(active=0, steps=sliders2, y=0, len=0.9,
                  currentvalue=dict(prefix="Año: "))],
    updatemenus=[dict(type='buttons', showactive=False, y=0.05, x=0.05,
                      buttons=[dict(label="▶ Reproducir",
                                    method="animate",
                                    args=[None,{"frame":{"duration":600},"fromcurrent":True}])])]
)
fig2.show()

### 8.6. Estados con mayor y menor mortalidad

In [6]:
stats_2017 = (estados_2017[estados_2017['cause_name']=='All causes']
               .nlargest(5,'age_adjusted_death_rate')[['state','age_adjusted_death_rate']])
low_2017 = (estados_2017[estados_2017['cause_name']=='All causes']
              .nsmallest(5,'age_adjusted_death_rate')[['state','age_adjusted_death_rate']])

fig = go.Figure()
fig.add_trace(go.Bar(x=stats_2017['state'], y=stats_2017['age_adjusted_death_rate'],
                     name='Mayor mortalidad', marker_color='#E63946',
                     text=stats_2017['age_adjusted_death_rate'].round(1), textposition='outside'))
fig.add_trace(go.Bar(x=low_2017['state'], y=low_2017['age_adjusted_death_rate'],
                     name='Menor mortalidad', marker_color='#457B9D',
                     text=low_2017['age_adjusted_death_rate'].round(1), textposition='outside'))
fig.update_layout(title='Estados con mayor y menor tasa ajustada de mortalidad (2017)',
                  yaxis_title='Tasa por 100,000 hab.', template='plotly_white',
                  height=430, barmode='group')
fig.show()
print("\nTop 5 mayor mortalidad (2017):")
print(stats_2017.to_string(index=False))
print("\nTop 5 menor mortalidad (2017):")
print(low_2017.to_string(index=False))


Top 5 mayor mortalidad (2017):
        state  age_adjusted_death_rate
West Virginia                    957.1
  Mississippi                    951.3
     Kentucky                    929.9
      Alabama                    917.7
     Oklahoma                    902.4

Top 5 menor mortalidad (2017):
      state  age_adjusted_death_rate
     Hawaii                    584.9
 California                    618.7
   New York                    623.6
Connecticut                    651.2
  Minnesota                    656.4


**Interpretación:** La comparación de tasas de mortalidad en los estados de EE. UU. para 2017 muestra una marcada desigualdad regional: mientras West Virginia, Mississippi, Kentucky, Alabama y Oklahoma presentan las tasas más altas, otros como Hawái, California, Nueva York, Connecticut y Minnesota registran las más bajas.